# Datasets

- 380,000 Restaurants Mostly USA Based [^1]
- Uber Eats USA Restaurants [^2]
- Yelp Restaurant Dataset [^3]

[^1]: <https://www.kaggle.com/datasets/kwxdata/380k-restaurants-mostly-usa-based?select=380K_US_Restaurants.csv>
[^2]: <https://www.kaggle.com/datasets/ahmedshahriarsakib/uber-eats-usa-restaurants-menus?select=restaurants.csv>
[^3]: <https://huggingface.co/datasets/jaimik69/Yelp-Restaurant-Dataset/tree/main>

In [1]:
# ==========================================
# 1) Imports & options (only required ones)
# ==========================================
import re, ast, json
import pandas as pd
from urllib.parse import urlparse
import usaddress

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# ==========================================
# 2) Canonical schema
# ==========================================
CANON_COLS = [
    "source", "source_id",
    "name", "name_norm",
    "website", "map_url",
    "phone_raw", "phone_e164",
    "address_line1", "address_line2",
    "street", "house_number",
    "city", "state", "postal_code", "country",
    "latitude", "longitude",
    "time_zone",
    "category_primary", "categories", "price_range", "price_level",
    "rating", "rating_count",
    "is_open",
    "hours_json",
]

# ==========================================
# 3) Helper functions (only the ones used)
# ==========================================
def clean_str(x: str) -> str:
    """Trim and collapse whitespace; return empty string for non-strings."""
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x

def strip_company_suffixes(name: str) -> str:
    """Remove common company suffixes like Inc., LLC, Co., etc."""
    if not name:
        return name
    name = re.sub(r"\b(inc|inc\.|llc|l\.l\.c\.|corp|corp\.|co|co\.|ltd|ltd\.)\b", "", name, flags=re.I)
    return re.sub(r"\s{2,}", " ", name).strip()

def normalize_name(name: str) -> str:
    """
    Normalize restaurant names WITHOUT removing generic words.
    - lowercase, diacritics -> ascii-ish
    - remove punctuation
    - remove company suffixes
    - collapse whitespace
    """
    if not isinstance(name, str) or not name.strip():
        return ""
    s = name.lower()
    s = (s
         .replace("é","e").replace("è","e").replace("ê","e").replace("á","a").replace("à","a")
         .replace("ö","o").replace("ü","u").replace("ä","a").replace("ß","ss"))
    s = re.sub(r"[^\w\s]", " ", s)
    s = strip_company_suffixes(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_url(url):
    """Return normalized URL (ensure scheme); None on failure/empty."""
    if not isinstance(url, str) or not url.strip():
        return None
    try:
        p = urlparse(url.strip())
        if not p.scheme:
            p = urlparse("http://" + url.strip())
        return p.geturl()
    except Exception:
        return None

def to_float(x):
    """Safe float conversion; None on failure."""
    try:
        return float(x)
    except Exception:
        return None

def to_int(x):
    """Safe int conversion; None on failure."""
    try:
        return int(x)
    except Exception:
        return None

def categories_to_list(x):
    """Robustly parse categories from list/CSV/JSON-like strings into a list[str]."""
    if x is None:
        return []
    if isinstance(x, list):
        return [clean_str(str(v)) for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        # try Python/JSON literals
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, list):
                    return [clean_str(str(v)) for v in obj if str(v).strip()]
                if isinstance(obj, dict):
                    # keep keys where value is truthy
                    return [k for k, v in obj.items() if v in (True, "True", "true", 1, "1")]
            except Exception:
                pass
        # fall back to common separators
        parts = re.split(r"[|,;/]+", s)
        return [clean_str(p) for p in parts if clean_str(p)]
    return [clean_str(str(x))] if str(x).strip() else []

def parse_us_address(addr: str, fallback_city=None, fallback_state=None, fallback_zip=None):
    """
    Parse US address into components using usaddress; fall back to simple heuristics.
    Returns: (address_line1, address_line2, street, house_number, city, state, postal_code, country)
    """
    addr = clean_str(addr)
    if not addr:
        return (None, None, None, None, fallback_city, fallback_state, fallback_zip, "US")
    try:
        tagged, _ = usaddress.tag(addr)
        house_number = tagged.get("AddressNumber")
        street_parts = [tagged.get("StreetNamePreType"), tagged.get("StreetNamePreDirectional"),
                        tagged.get("StreetName"), tagged.get("StreetNamePostType"), tagged.get("StreetNamePostDirectional")]
        street = " ".join([p for p in street_parts if p and p.strip()]) or None
        address_line1 = " ".join([p for p in [house_number, street] if p])
        city = tagged.get("PlaceName", fallback_city)
        state = tagged.get("StateName", fallback_state)
        postal_code = tagged.get("ZipCode", fallback_zip)
        add2_parts = [tagged.get("OccupancyType"), tagged.get("OccupancyIdentifier")]
        address_line2 = " ".join([p for p in add2_parts if p and str(p) != "None"]) or None
        return (address_line1 or None, address_line2, street, house_number, city, state, postal_code, "US")
    except Exception:
        # naive comma-based parsing as a fallback
        parts = [p.strip() for p in addr.split(",")]
        city = fallback_city
        state = fallback_state
        postal_code = fallback_zip
        if len(parts) >= 3:
            city = parts[-3]
            st_zip = parts[-2]
            m = re.search(r"([A-Z]{2})\s+(\d{5})(?:-\d{4})?$", st_zip)
            if m:
                state, postal_code = m.group(1), m.group(2)
        address_line1 = parts[0] if parts else None
        m2 = re.match(r"(\d+)\s+(.*)", address_line1 or "")
        house_number = m2.group(1) if m2 else None
        street = m2.group(2) if m2 else None
        return (address_line1, None, street, house_number, city, state, postal_code, "US")

def safe_literal_eval_dict(x):
    """Safely parse a string that may contain a Python dict literal; return dict or None."""
    if not isinstance(x, str):
        return None
    try:
        obj = ast.literal_eval(x)
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

# ==========================================
# 4) Dataset-specific transformers
# ==========================================
def transform_ds1(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Transform: 380k restaurants (mostly USA)."""
    df = df_raw.copy()

    # rename columns to canonical names when present
    rename_map = {
        "Title": "name",
        "Link": "map_url",
        "Category": "category_primary",
        "Rating": "rating",
        "Website": "website",
        "Phone": "phone_raw",
        "Address": "address_raw",
        "Images": "images",
        "Categories": "categories_raw",
        "Geo_Coordinates": "geo",
        "Time_Zone": "time_zone",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Latitude & Longitude": "latlong",
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    # latitude/longitude from dict-like "geo" if needed
    if "geo" in df.columns:
        geo_parsed = df["geo"].apply(safe_literal_eval_dict)
        if "latitude" not in df.columns:
            df["latitude"]  = geo_parsed.apply(lambda d: d.get("latitude")  if isinstance(d, dict) else None)
        if "longitude" not in df.columns:
            df["longitude"] = geo_parsed.apply(lambda d: d.get("longitude") if isinstance(d, dict) else None)

    # fallback: parse "latlong" text for lat/lon
    if "latlong" in df.columns and ("latitude" not in df or "longitude" not in df):
        ll = df["latlong"].astype(str)
        mlat = ll.str.extract(r"lat(?:itude)?[:=]\s*([\-0-9\.]+)", expand=False)
        mlon = ll.str.extract(r"lon(?:gitude)?[:=]\s*([\-0-9\.]+)", expand=False)
        if "latitude" not in df:  df["latitude"]  = mlat
        if "longitude" not in df: df["longitude"] = mlon

    # categories
    if "categories_raw" in df.columns:
        df["categories_list"] = df["categories_raw"].apply(categories_to_list)
    else:
        df["categories_list"] = df.get("category_primary", "").apply(categories_to_list)
    if "category_primary" not in df.columns:
        df["category_primary"] = None
    df["category_primary"] = df["category_primary"].fillna(df["categories_list"].apply(lambda x: x[0] if x else None))
    df["categories"] = df["categories_list"]

    # address parsing
    addr_series = df["address_raw"] if "address_raw" in df.columns else pd.Series([None]*len(df))
    addr_tuple = addr_series.apply(lambda a: parse_us_address(a))
    (df["address_line1"], df["address_line2"], df["street"], df["house_number"],
     df["city"], df["state"], df["postal_code"], df["country"]) = zip(*addr_tuple)

    # names & URLs
    df["name"] = df.get("name", "").apply(clean_str)
    df["name_norm"] = df["name"].apply(normalize_name)
    df["website"] = df.get("website").apply(normalize_url) if "website" in df.columns else None
    df["map_url"] = df.get("map_url").apply(normalize_url)

    # phone: pass through raw; do not normalize
    df["phone_raw"] = df.get("phone_raw")
    df["phone_e164"] = None

    # price: do not derive (keep both None)
    df["price_level"] = None
    df["price_range"] = None

    # ratings
    df["rating"] = df.get("rating").apply(to_float)
    df["rating_count"] = None

    # geo
    df["latitude"]  = df.get("latitude").apply(to_float)
    df["longitude"] = df.get("longitude").apply(to_float)

    # source & id
    df["source"] = "kaggle_380k"
    df["source_id"] = df["map_url"].fillna(df["name"] + "|" + df["postal_code"].fillna(""))

    # hours / open / timezone
    df["hours_json"] = None
    df["is_open"] = None
    df["time_zone"] = df.get("time_zone")

    # finalize
    for col in CANON_COLS:
        if col not in df.columns:
            df[col] = None
    return df[CANON_COLS]

def transform_ds2(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Transform: Uber Eats USA Restaurants (expects columns like):
    id, name, score, ratings, category, price_range, full_address, zip_code, lat, lng
    """
    df = df_raw.copy()

    # flexible renaming, including lng -> longitude
    rename_map = {
        "id": "source_id",
        "name": "name",
        "score": "rating",
        "ratings": "rating_count",
        "category": "categories_raw",
        "price_range": "price_range",
        "full_address": "address_raw",
        "zip_code": "postal_code",
        "lat": "latitude",
        "lng": "longitude",
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    # ensure required columns exist
    for col in ["source_id","name","rating","rating_count","categories_raw","price_range",
                "address_raw","postal_code","latitude","longitude"]:
        if col not in df.columns:
            df[col] = None

    # categories
    df["categories_list"] = df.get("categories_raw").apply(categories_to_list)
    df["category_primary"] = df["categories_list"].apply(lambda x: x[0] if x else None)
    df["categories"] = df["categories_list"]

    # address parsing
    addr_tuple = df.apply(lambda r: parse_us_address(
        r.get("address_raw"),
        fallback_city=None, fallback_state=None,
        fallback_zip=str(r.get("postal_code")) if pd.notna(r.get("postal_code")) else None
    ), axis=1)
    (df["address_line1"], df["address_line2"], df["street"], df["house_number"],
     df["city"], df["state"], df["postal_code"], df["country"]) = zip(*addr_tuple)

    # names & URLs
    df["name"] = df["name"].apply(clean_str)
    df["name_norm"] = df["name"].apply(normalize_name)
    df["website"] = None
    df["map_url"] = None

    # phones: not present -> keep None
    df["phone_raw"] = None
    df["phone_e164"] = None

    # price: keep raw symbol if present; do not compute level
    df["price_range"] = df["price_range"]
    df["price_level"] = None

    # ratings
    df["rating"] = df["rating"].apply(to_float)
    df["rating_count"] = df["rating_count"].apply(to_int)

    # geo
    df["latitude"]  = df["latitude"].apply(to_float)
    df["longitude"] = df["longitude"].apply(to_float)

    # hours / open / timezone
    df["time_zone"] = None
    df["hours_json"] = None
    df["is_open"] = None

    # source
    df["source"] = "uber_eats"

    # finalize
    for col in CANON_COLS:
        if col not in df.columns:
            df[col] = None
    return df[CANON_COLS]

def transform_ds3_yelp_new(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Transform: NEW Yelp CSV (yelp.csv)
    Expected columns:
      restaurant_id, name, cuisines, price, rating, review_count, phone, url,
      address, city, state, zip_code, country, latitude, longitude, source
    """
    df = df_raw.copy()

    # rename into canonical / staging fields
    rename_map = {
        "restaurant_id": "source_id",
        "name": "name",
        "cuisines": "categories_raw",
        "price": "price_raw",            # keep as-is; no level derivation
        "rating": "rating",
        "review_count": "rating_count",
        "phone": "phone_raw",            # pass-through
        "url": "map_url",                # Yelp business URL -> map_url
        "address": "address_line1",
        "city": "city",
        "state": "state",
        "zip_code": "postal_code",
        "country": "country",
        "latitude": "latitude",
        "longitude": "longitude",
        "source": "src_in_file",         # optional; typically 'yelp'
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df.rename(columns={k: v}, inplace=True)

    # ensure required columns exist
    required = ["source_id","name","categories_raw","price_raw","rating","rating_count",
                "phone_raw","map_url","address_line1","city","state","postal_code",
                "country","latitude","longitude","src_in_file"]
    for col in required:
        if col not in df.columns:
            df[col] = None

    # categories
    df["categories_list"] = df.get("categories_raw").apply(categories_to_list)
    df["category_primary"] = df["categories_list"].apply(lambda x: x[0] if x else None)
    df["categories"] = df["categories_list"]

    # address parsing with fallbacks from structured fields
    addr_tuple = df.apply(lambda r: parse_us_address(
        r.get("address_line1"),
        fallback_city=r.get("city"),
        fallback_state=r.get("state"),
        fallback_zip=str(r.get("postal_code")) if pd.notna(r.get("postal_code")) else None
    ), axis=1)
    (df["address_line1"], df["address_line2"], df["street"], df["house_number"],
     df["city"], df["state"], df["postal_code"], df["country_from_parse"]) = zip(*addr_tuple)
    # prefer provided country if present
    df["country"] = df["country"].fillna(df["country_from_parse"])
    df.drop(columns=["country_from_parse"], inplace=True)

    # names & URLs
    df["name"] = df["name"].apply(clean_str)
    df["name_norm"] = df["name"].apply(normalize_name)
    df["website"] = None
    df["map_url"] = df["map_url"].apply(normalize_url)

    # phone: keep raw; do not normalize
    df["phone_e164"] = None

    # price: keep raw into price_range; do not compute level
    df["price_range"] = df["price_raw"]
    df["price_level"] = None

    # ratings
    df["rating"] = df["rating"].apply(to_float)
    df["rating_count"] = df["rating_count"].apply(to_int)

    # geo
    df["latitude"]  = df["latitude"].apply(to_float)
    df["longitude"] = df["longitude"].apply(to_float)

    # hours / open / timezone
    df["is_open"] = None
    df["hours_json"] = None
    df["time_zone"] = None

    # source
    df["source"] = df.get("src_in_file").fillna("yelp").replace("", "yelp")

    # finalize
    for col in CANON_COLS:
        if col not in df.columns:
            df[col] = None
    return df[CANON_COLS]

# ==========================================
# 5) Orchestration: LOCAL CSVs only
# ==========================================
def prepare_all(
    ds1_local_csv: str,
    ds2_local_csv: str,
    ds3_local_csv: str,  # new Yelp CSV
):
    """Load three local CSVs and return harmonized DataFrames for downstream integration."""
    df1_raw = pd.read_csv(ds1_local_csv)
    df2_raw = pd.read_csv(ds2_local_csv)
    df3_raw = pd.read_csv(ds3_local_csv)

    df1 = transform_ds1(df1_raw)
    df2 = transform_ds2(df2_raw)
    df3 = transform_ds3_yelp_new(df3_raw)
    return df1, df2, df3

In [2]:
df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean = prepare_all(
    ds1_local_csv="datasets/380K_US_Restaurants.csv",
    ds2_local_csv="datasets/uber-eats-restaurants.csv",
    ds3_local_csv="datasets/yelp.csv",
)

C:\Users\Gregor Debus\AppData\Local\Temp\ipykernel_16292\3124994922.py:446: DtypeWarning: Columns (3,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df1_raw = pd.read_csv(ds1_local_csv)


In [3]:
df1_380k_clean.head(1)

,source,source_id,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,time_zone,category_primary,categories,price_range,price_level,rating,rating_count,is_open,hours_json
0,kaggle_380k,https://www.google.com/maps/place/Dairy+Queen+...,Dairy Queen Grill & Chill,dairy queen grill chill,http://www.fourteenfoods.com/?y_source=1_ODk5N...,https://www.google.com/maps/place/Dairy+Queen+...,+1256-496-0404,None,3143 US-280,None,US-280,3143,Alexander City,AL,35010,US,32.93387,-85.970419,America/Chicago,Fast food restaurant,"[Fast food restaurant, Ice cream shop]",None,None,3.8,None,None,None


In [4]:
df2_uber_eats_clean.head(1)

,source,source_id,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,time_zone,category_primary,categories,price_range,price_level,rating,rating_count,is_open,hours_json
0,uber_eats,1,PJ Fresh (224 Daniel Payne Drive),pj fresh 224 daniel payne drive,None,None,None,None,224 Daniel Payne Drive,None,Daniel Payne Drive,224,Birmingham,AL,35207,US,33.562365,-86.830703,None,Burgers,"[Burgers, American, Sandwiches]",$,None,NaN,NaN,None,None


In [5]:
df3_yelp_clean.head(1)

,source,source_id,name,name_norm,website,map_url,phone_raw,phone_e164,address_line1,address_line2,street,house_number,city,state,postal_code,country,latitude,longitude,time_zone,category_primary,categories,price_range,price_level,rating,rating_count,is_open,hours_json
0,yelp,4MRM9BCU4x3E4e5xDRTjEQ,The Quimby,the quimby,None,https://www.yelp.com/biz/the-quimby-queens?adj...,1.347567e+10,None,135-25 142nd St,None,142nd St,135-25,Queens,NY,11436,US,40.66735,-73.79692,None,breakfast,[breakfast],NaN,None,4.2,21,None,None


In [6]:
# Export to parquet files
import os

def export_parquet(dfs, names, out_dir="out_parquet"):
    """Write each DataFrame to <out_dir>/<name>.parquet (one-to-one with names)."""
    if len(dfs) != len(names):
        raise ValueError(f"Length mismatch: dfs={len(dfs)} vs names={len(names)}")
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for df, n in zip(dfs, names):
        path = os.path.join(out_dir, f"{n}.parquet")
        df.to_parquet(path, index=False)
        paths.append(path)
    print(f"Wrote {len(paths)} files to {out_dir}/")
    return paths

export_parquet(
    [df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean],
    ["ds1_kaggle380k_clean", "ds2_ubereats_clean", "ds3_yelp_clean"]
)

Wrote 3 files to out_parquet/


['out_parquet\\ds1_kaggle380k_clean.parquet',
 'out_parquet\\ds2_ubereats_clean.parquet',
 'out_parquet\\ds3_yelp_clean.parquet']

In [7]:
import math
import pandas as pd

# ---------- Geo helpers ----------
def _haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km; returns +inf if any coord is missing."""
    try:
        if None in (lat1, lon1, lat2, lon2):
            return float("inf")
        rlat1, rlon1, rlat2, rlon2 = map(math.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = rlat2 - rlat1, rlon2 - rlon1
        a = math.sin(dlat/2)**2 + math.cos(rlat1)*math.cos(rlat2)*math.sin(dlon/2)**2
        return 6371.0088 * (2 * math.asin(math.sqrt(a)))
    except Exception:
        return float("inf")

def _auto_decimals(tol_meters: float) -> int:
    """Choose rounding precision for candidate pruning based on tolerance."""
    if tol_meters <= 30:   return 4   # ~11 m
    if tol_meters <= 300:  return 3   # ~111 m
    if tol_meters <= 3000: return 2   # ~1.1 km
    return 1                          # ~11 km

def _prep_temp(df: pd.DataFrame, decimals: int):
    """
    Small projection (source, source_id, lat, lon, rowid, rounded lat/lon).
    Keeps the ORIGINAL row index in 'rowid' for traceability.
    """
    mask = df["latitude"].notna() & df["longitude"].notna()
    tmp = df.loc[mask, ["source", "source_id", "latitude", "longitude"]].copy()
    tmp["rowid"] = df.index[mask]
    tmp["lat_r"] = tmp["latitude"].round(decimals)
    tmp["lon_r"] = tmp["longitude"].round(decimals)
    return tmp

def _pairwise_geo_matches(dfA: pd.DataFrame, dfB: pd.DataFrame, tol_meters: float, decimals: int = None):
    """
    Find matches between dfA and dfB if distance <= tol_meters.
    Uses rounded coord join for candidates + exact Haversine filter.
    Returns (n_matches, matches_df).
    """
    tol_km = tol_meters / 1000.0
    if decimals is None:
        decimals = _auto_decimals(tol_meters)

    A = _prep_temp(dfA, decimals)
    B = _prep_temp(dfB, decimals)

    # Candidate pairs via rounded grid
    cand = A.merge(B, on=["lat_r", "lon_r"], suffixes=("_a", "_b"))
    if cand.empty:
        return 0, cand

    # Exact distance filter
    dists = cand.apply(lambda r: _haversine_km(r["latitude_a"], r["longitude_a"],
                                               r["latitude_b"], r["longitude_b"]), axis=1)
    cand = cand.loc[dists <= tol_km].copy()

    # De-duplicate pairs by original row ids
    cand = cand.drop_duplicates(subset=["rowid_a", "rowid_b"])

    # Compact output
    out = cand[[
        "source_a","source_id_a","latitude_a","longitude_a","rowid_a",
        "source_b","source_id_b","latitude_b","longitude_b","rowid_b"
    ]].reset_index(drop=True)

    return len(out), out

# ---------- Components across 3 datasets ----------
class _DSU:
    """Union-Find to build connected components from pairwise edges."""
    def __init__(self): self.p, self.r = {}, {}
    def find(self, x):
        if x not in self.p: self.p[x]=x; self.r[x]=0; return x
        while x != self.p[x]:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1

def _gid(source, source_id, rowid):
    """Stable node id even if source_id is missing/duplicated."""
    return f"{source}::{'' if source_id is None else str(source_id)}::{str(rowid)}"

def geo_overlap_summary_3(df1: pd.DataFrame, df2: pd.DataFrame, df3: pd.DataFrame,
                          tol_meters: float = 50.0, decimals: int = None):
    """
    Find geo-based matches across THREE harmonized DataFrames.
    Returns (summary_dict, matches_dict).
    - summary_dict has pairwise counts and number of components spanning all 3.
    - matches_dict holds the pairwise match DataFrames.
    """
    # All 3 pairs
    n12, m12 = _pairwise_geo_matches(df1, df2, tol_meters, decimals)
    n13, m13 = _pairwise_geo_matches(df1, df3, tol_meters, decimals)
    n23, m23 = _pairwise_geo_matches(df2, df3, tol_meters, decimals)

    # Build components from confirmed pairwise matches
    dsu = _DSU()
    def _union_all(md):
        if md is None or md.empty: return
        for _, r in md.iterrows():
            a = _gid(r["source_a"], r["source_id_a"], r["rowid_a"])
            b = _gid(r["source_b"], r["source_id_b"], r["rowid_b"])
            dsu.union(a, b)

    for md in (m12, m13, m23): _union_all(md)

    # Collect nodes and form components
    nodes = set()
    for md in (m12, m13, m23):
        if md is None or md.empty: continue
        for _, r in md.iterrows():
            nodes.add(_gid(r["source_a"], r["source_id_a"], r["rowid_a"]))
            nodes.add(_gid(r["source_b"], r["source_id_b"], r["rowid_b"]))

    comps = {}
    for n in nodes:
        root = dsu.find(n)
        comps.setdefault(root, set()).add(n)

    # Count how many distinct sources each component spans
    def _src_of(g): return g.split("::", 1)[0]
    span_counts = [len({_src_of(g) for g in group}) for group in comps.values()]
    three_way = sum(1 for s in span_counts if s >= 3)

    summary = {
        "tolerance_meters": tol_meters,
        "pairwise_matches": {"1_vs_2": n12, "1_vs_3": n13, "2_vs_3": n23},
        "three_way_overlap_components": three_way,
        "total_components_linked_by_geo": len(comps),
    }
    matches = {"1_vs_2": m12, "1_vs_3": m13, "2_vs_3": m23}
    return summary, matches

In [9]:
summary_geo, matches_geo = geo_overlap_summary_3(
    df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean,
    tol_meters=25  # adjust if needed (e.g., 25, 100, 200)
)
print(summary_geo)

{'tolerance_meters': 25, 'pairwise_matches': {'1_vs_2': 5102, '1_vs_3': 3828, '2_vs_3': 660}, 'three_way_overlap_components': 19, 'total_components_linked_by_geo': 3829}


In [10]:
import re
import pandas as pd

# ---------- Address normalizers ----------
_DIR_TOKENS = {"n","s","e","w","ne","nw","se","sw","north","south","east","west"}
_SFX_MAP = {
    "street":"st","st":"st","st.":"st",
    "road":"rd","rd":"rd","rd.":"rd",
    "avenue":"ave","ave":"ave","ave.":"ave",
    "boulevard":"blvd","blvd":"blvd","blvd.":"blvd",
    "lane":"ln","ln":"ln","ln.":"ln",
    "drive":"dr","dr":"dr","dr.":"dr",
    "court":"ct","ct":"ct","ct.":"ct",
    "circle":"cir","cir":"cir","cir.":"cir",
    "place":"pl","pl":"pl","pl.":"pl",
    "parkway":"pkwy","pkwy":"pkwy","pkwy.":"pkwy","pkway":"pkwy",
    "highway":"hwy","hwy":"hwy","hwy.":"hwy",
    "terrace":"ter","ter":"ter","ter.":"ter",
    "way":"way","square":"sq","sq":"sq","sq.":"sq",
    "trail":"trl","trl":"trl","trl.":"trl",
    "alley":"aly","aly":"aly","aly.":"aly"
}

def _zip5(z):
    if z is None: return None
    m = re.search(r"(\d{5})", str(z))
    return m.group(1) if m else None

def _norm_state(s):
    return str(s).strip().upper() if s is not None and str(s).strip() else None

def _norm_city(s):
    if s is None: return None
    s = re.sub(r"[^\w\s]", " ", str(s).lower())
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def _norm_house(h):
    if h is None: return None
    s = re.sub(r"[^0-9a-z]", "", str(h).strip().lower())  # keep 123a
    return s or None

def _norm_street(street):
    if street is None: return None
    s = re.sub(r"[^\w\s]", " ", str(street).lower())
    toks = [t for t in re.split(r"\s+", s) if t]
    out = []
    for t in toks:
        if t in _DIR_TOKENS:   # drop N/S/E/W tokens
            continue
        out.append(_SFX_MAP.get(t, t))  # normalize suffixes
    s = " ".join(out).strip()
    return s or None

def _fallback_from_line1(line1):
    """If street/house are missing, try a quick split from address_line1."""
    if not isinstance(line1, str) or not line1.strip():
        return (None, None)
    m = re.match(r"\s*([0-9A-Za-z\-]+)\s+(.*)", line1.strip())
    if not m:
        return (None, _norm_street(line1))
    return (_norm_house(m.group(1)), _norm_street(m.group(2)))

def _prep_addr_temp(df: pd.DataFrame):
    """
    Build a small projection with normalized address parts.
    Does NOT modify the original dataframe.
    """
    cols = ["source","source_id","address_line1","street","house_number","city","state","postal_code"]
    tmp = df[cols].copy()

    # Fill missing street/house from address_line1 when possible; normalize both ways
    fill = tmp.apply(
        lambda r: _fallback_from_line1(r["address_line1"])
                  if (pd.isna(r["street"]) or pd.isna(r["house_number"]))
                  else (_norm_house(r["house_number"]), _norm_street(r["street"])),
        axis=1
    )
    tmp["house_norm"], tmp["street_norm"] = zip(*fill)

    tmp["city_norm"] = tmp["city"].apply(_norm_city)
    tmp["state_u"]   = tmp["state"].apply(_norm_state)
    tmp["zip5"]      = tmp["postal_code"].apply(_zip5)

    tmp["rowid"] = df.index  # preserve original index
    return tmp

# ---------- Pairwise matching by address ----------
def _pairwise_address_matches(dfA: pd.DataFrame, dfB: pd.DataFrame):
    """
    Return (n_matches, matches_df) where matches require:
      - house_norm and street_norm equal, and
      - (zip5 equal when both present) OR (city_norm + state_u equal).
    Uses exact matches on normalized fields.
    """
    A = _prep_addr_temp(dfA)
    B = _prep_addr_temp(dfB)

    # Filter rows with essential parts
    A1 = A.dropna(subset=["house_norm","street_norm"])
    B1 = B.dropna(subset=["house_norm","street_norm"])

    pairs = []

    # Path 1: ZIP available on both sides -> join on (zip5, house_norm, street_norm)
    A_zip = A1.dropna(subset=["zip5"])
    B_zip = B1.dropna(subset=["zip5"])
    p_zip = A_zip.merge(
        B_zip,
        on=["zip5","house_norm","street_norm"],
        suffixes=("_a","_b"),
        how="inner"
    )
    pairs.append(p_zip)

    # Path 2: Fallback when ZIP missing -> join on (city_norm, state_u, house_norm, street_norm)
    A_cs = A1.dropna(subset=["city_norm","state_u"])
    B_cs = B1.dropna(subset=["city_norm","state_u"])
    p_cs = A_cs.merge(
        B_cs,
        on=["city_norm","state_u","house_norm","street_norm"],
        suffixes=("_a","_b"),
        how="inner"
    )
    pairs.append(p_cs)

    cand = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()
    if cand.empty:
        return 0, cand

    # De-duplicate by original rows
    cand = cand.drop_duplicates(subset=["rowid_a","rowid_b"])

    # Compact view for inspection
    out = cand[[
        "source_a","source_id_a","address_line1_a","city_a","state_a","postal_code_a","rowid_a",
        "source_b","source_id_b","address_line1_b","city_b","state_b","postal_code_b","rowid_b",
    ]].reset_index(drop=True)

    return len(out), out

# ---------- Build components across 3 datasets ----------
class _DSU:
    """Union-Find to build connected components from pairwise edges."""
    def __init__(self): self.p, self.r = {}, {}
    def find(self, x):
        if x not in self.p: self.p[x]=x; self.r[x]=0; return x
        while x != self.p[x]:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1

def _gid(source, source_id, rowid):
    """Stable node id even if source_id is missing/duplicated."""
    return f"{source}::{'' if source_id is None else str(source_id)}::{str(rowid)}"

def address_overlap_summary_3(df1: pd.DataFrame, df2: pd.DataFrame, df3: pd.DataFrame):
    """
    Count address-based matches across THREE harmonized DataFrames:
      - Pairwise counts for (1↔2), (1↔3), (2↔3)
      - Number of connected components spanning all 3 sources
    Matching uses normalized exact rules in _pairwise_address_matches.
    """
    n12, m12 = _pairwise_address_matches(df1, df2)
    n13, m13 = _pairwise_address_matches(df1, df3)
    n23, m23 = _pairwise_address_matches(df2, df3)

    # Build components
    dsu = _DSU()
    def _union_all(md):
        if md is None or md.empty: return
        for _, r in md.iterrows():
            a = _gid(r["source_a"], r["source_id_a"], r["rowid_a"])
            b = _gid(r["source_b"], r["source_id_b"], r["rowid_b"])
            dsu.union(a, b)
    for md in (m12, m13, m23): _union_all(md)

    # Collect nodes and form components
    nodes = set()
    for md in (m12, m13, m23):
        if md is None or md.empty: continue
        for _, r in md.iterrows():
            nodes.add(_gid(r["source_a"], r["source_id_a"], r["rowid_a"]))
            nodes.add(_gid(r["source_b"], r["source_id_b"], r["rowid_b"]))

    comps = {}
    for n in nodes:
        root = dsu.find(n)
        comps.setdefault(root, set()).add(n)

    # Count components that span all three sources
    def _src_of(g): return g.split("::", 1)[0]
    span_counts = [len({_src_of(g) for g in group}) for group in comps.values()]
    three_way = sum(1 for s in span_counts if s >= 3)

    summary = {
        "pairwise_matches": {"1_vs_2": n12, "1_vs_3": n13, "2_vs_3": n23},
        "three_way_overlap_components": three_way,
        "total_components_linked_by_address": len(comps),
    }
    matches = {"1_vs_2": m12, "1_vs_3": m13, "2_vs_3": m23}
    return summary, matches

In [11]:
summary_addr, matches_addr = address_overlap_summary_3(
    df1_380k_clean, df2_uber_eats_clean, df3_yelp_clean
)
print(summary_addr)

# Inspect some 1↔3 address matches
m13 = matches_addr["1_vs_3"]
if not m13.empty:
    display(m13.head(10))

{'pairwise_matches': {'1_vs_2': 25789, '1_vs_3': 12295, '2_vs_3': 5705}, 'three_way_overlap_components': 151, 'total_components_linked_by_address': 10382}


,source_a,source_id_a,address_line1_a,city_a,state_a,postal_code_a,rowid_a,source_b,source_id_b,address_line1_b,city_b,state_b,postal_code_b,rowid_b
0,kaggle_380k,https://www.google.com/maps/place/The+Iberian+...,121 Sycamore St,Decatur,GA,30030,897,yelp,if4TMfn8v4GgOm-aKAUwrg,121 Sycamore St,Decatur,GA,30030,6370
1,kaggle_380k,https://www.google.com/maps/place/The+White+Bu...,123 E Court Square,Decatur,GA,30030,901,yelp,B1utosVn_70fiRUxJQRKgA,123 Court E Square,Decatur,GA,30030,6344
2,kaggle_380k,https://www.google.com/maps/place/The+Deer+and...,155 Sycamore St,Decatur,GA,30030,902,yelp,W6unkmXcb8z6iYDWzHNEMQ,155 Sycamore St,Decatur,GA,30030,6354
3,kaggle_380k,https://www.google.com/maps/place/Wheelhouse+C...,1479 Scott Blvd,Decatur,GA,30030,924,yelp,U4K4dCTtGeNNPUtDaC3HHg,1479 Scott Blvd,Decatur,GA,30030,6350
4,kaggle_380k,https://www.google.com/maps/place/Cap%27t+Loui...,319 W Ponce de Leon Ave,Decatur,GA,30030,945,yelp,Z15YtPh0bK8kKU6-F902jQ,319 W Ponce De Leon Ave,Decatur,GA,30030,6357
5,kaggle_380k,https://www.google.com/maps/place/bb.q+Chicken...,319 W Ponce de Leon Ave,Decatur,GA,30030,958,yelp,Z15YtPh0bK8kKU6-F902jQ,319 W Ponce De Leon Ave,Decatur,GA,30030,6357
6,kaggle_380k,https://www.google.com/maps/place/The+Po%27Boy...,1369 Clairmont Rd,Decatur,GA,30033,982,yelp,opGvvF6iQ_Ms5xO_MxnuUQ,1369 Clairmont Rd,Decatur,GA,30033,6376
7,kaggle_380k,https://www.google.com/maps/place/Claim+Jumper...,10125 W McDowell Rd,Avondale,AZ,85392,3520,yelp,3rSqoiZkaGBeLEEqpULybw,10125 W McDowell Rd,Avondale,AZ,85392,8124
8,kaggle_380k,https://www.google.com/maps/place/Flavors+of+L...,13025 W Rancho Santa Fe Blvd,Avondale,AZ,85392,3521,yelp,8Vpfy_WtcVvZS8Wfz7JQJA,13025 W Rancho Santa Fe Blvd,Avondale,AZ,85392,8119
9,kaggle_380k,https://www.google.com/maps/place/Islands+Rest...,10055 W McDowell Rd,Avondale,AZ,85392,3522,yelp,YvXUrRD3Rjj4SpFuqbmUfg,10055 W McDowell Rd,Avondale,AZ,85392,8225
